In [1]:
import QuantLib as ql

In [2]:
today = ql.Date(8, ql.October, 2014)
ql.Settings.instance().evaluationDate = today

<b>A somewhat exotic option</b><br/>
As an example, we’ll use a knock-in barrier option:

In [3]:
option = ql.BarrierOption(ql.Barrier.UpIn,
                         120.0, #barrier
                         0.0, #rebate
                         ql.PlainVanillaPayoff(ql.Option.Call, 100.0),
                         ql.EuropeanExercise(ql.Date(8, ql.January, 2015)))

In [4]:
u = ql.SimpleQuote(100.0)
r = ql.SimpleQuote(0.01)
sigma = ql.SimpleQuote(0.20)

from the Quotes we build the flat curves

riskFreeCurve = ql.FlatForward(0, ql.TARGET(),
                              ql.QuoteHandle(r), ql.Actual360())
volatility = ql.BlackConstantVol(0, ql.TARGET(),
                                ql.QuoteHandle(sigma), ql.Actual360())

In [6]:
process = ql.BlackScholesProcess(ql.QuoteHandle(u),
                                ql.YieldTermStructureHandle(riskFreeCurve),
                                 ql.BlackVolTermStructureHandle(volatility))

Finally, we build the engine (the library provides one based on an analytic formula) and set it to the option.

In [7]:
option.setPricingEngine(ql.AnalyticBarrierEngine(process))

Now we can ask the option for its value…

In [8]:
print(option.NPV())

1.3657980739109867


…but we’re not so lucky when it comes to Greeks:

In [10]:
print(option.delta()) RuntimeError: delta not provided

<h4>Numerical calculation</h4>

We can use numerical differentiation to approximate the Greeks

In [11]:
u0 = u.value(); h = 0.01

In [12]:
P0 = option.NPV() ; print(P0)

1.3657980739109867


we increase the underlying value and get the new option value…

In [14]:
u.setValue(u0 + h)
P_plus = option.NPV(); print(P_plus)

1.3688112201958078


…then we do the same after decreasing the underlying value.

In [15]:
u.setValue(u0 - h)
P_minus = option.NPV(); print(P_minus)

1.3627900998610203


Finally, we set the underlying value back to its current value.

In [16]:
u.setValue(u0)

Applying the formulas above give us the desired Greeks:

In [17]:
Delta = (P_plus - P_minus) / (2*h)
Gamma = (P_plus - 2*P0 + P_minus)/(h*h)
print(Delta)
print(Gamma)

0.3010560167393761
0.05172234854633473


Can calculate Rho and Vega

In [18]:
r0 = r.value(); h = 0.0001
r.setValue(r0+h); P_plus = option.NPV()
r.setValue(r0)
Rho = (P_plus - P0)/h; print(Rho)

6.531038494281827


In [20]:
sigma0= sigma.value(); h = 0.0001
sigma.setValue(sigma0+h); P_plus = option.NPV()
sigma.setValue(sigma0)
Vega = (P_plus - P0)/h; print(Vega)

26.52519924198904


The approach for the Theta is a bit different, although it still relies on the fact that the option reacts
to the change in the market data. The problem is that we don’t have the time to maturity available
as a quote, as was the case for the other quantities. Instead, since we set up the term structures
so that they move with the evaluation date, we just have to set it to tomorrow’s date to get the
corresponding option value:

In [21]:
ql.Settings.instance().evaluationDate= today + 1
P1= option.NPV()
h = 1.0/365
Theta=(P1-P0)/h; print(Theta)

-10.770888399441302
